# Notebook 03 -- Category Harmonization



> **Note:** This notebook is pre-run. Data lives at `C:\BA_Data\` and is not included in this repository.

---

This notebook documents the category harmonization step. The core problem is that the DSA API changed its `category` vocabulary on July 1, 2025 (v1 to v2), making direct comparison of records across that boundary impossible without a mapping layer.

In [44]:
from pathlib import Path

import polars as pl

DATA_ROOT = Path(r"C:\BA_Data")
TIKTOK_RAW = DATA_ROOT / "tiktok_de_2025.parquet"
X_RAW = DATA_ROOT / "x_de_2025.parquet"
TIKTOK_HARMONIZED = DATA_ROOT / "tiktok_de_2025_harmonized.parquet"
X_HARMONIZED = DATA_ROOT / "x_de_2025_harmonized.parquet"

# Mapping table is part of the repository
REPO_ROOT = Path("__file__").resolve().parents[1] if "__file__" in dir() else Path(".").resolve().parent
MAPPING_CSV = REPO_ROOT / "03_harmonization" / "taxonomy_mapping_v1_v2.csv"

## 1. The v1/v2 Schema Break

On **July 1, 2025** the DSA Commission released a new submission API (v2) that modified the `category` field:

- Some categories were **renamed** (e.g. `SCOPE_OF_PLATFORM_SERVICE` became `OTHER_VIOLATION_TC`)
- One category was **removed** (`NON_CONSENSUAL_BEHAVIOUR`, no v2 equivalent)
- One category was officially **introduced** in v2 (`CYBER_VIOLENCE`), though it already appeared in some v1 records

[DSA Official change logs](https://transparency.dsa.ec.europa.eu/page/api-documentation#changelog)

Without harmonization, any category-level comparison between the v1 period (Jan--Jun 2025) and the v2 period (Jul--Dec 2025) would be misleading.

In [45]:
# Raw category labels in TikTok, split by api_version
raw_cats = (
    pl.scan_parquet(str(TIKTOK_RAW))
    .group_by([ "category","api_version",])
    .agg(pl.len().alias("count"))
    .sort(["category", "count"], descending=[False, True])
    .collect(engine="streaming")
)

print("TikTok raw categories per api_version:")
with pl.Config(fmt_str_lengths=100, tbl_rows=100):
    display(raw_cats)

KeyboardInterrupt: 

In [ ]:
# Raw category labels in X, split by api_version
raw_cats_x = (
    pl.scan_parquet(str(X_RAW))
    .group_by([ "category","api_version",])
    .agg(pl.len().alias("count"))
    .sort(["category", "count"], descending=[False, True])
    .collect()
)

print("X raw categories per api_version:")
with pl.Config(fmt_str_lengths=100, tbl_rows=100):
    display(raw_cats_x)

X raw categories per api_version:


category,api_version,count
str,str,u32
"""STATEMENT_CATEGORY_ANIMAL_WELFARE""","""v1""",10
"""STATEMENT_CATEGORY_ANIMAL_WELFARE""","""v2""",3
"""STATEMENT_CATEGORY_CYBER_VIOLENCE""","""v2""",19143
"""STATEMENT_CATEGORY_CYBER_VIOLENCE""","""v1""",34
"""STATEMENT_CATEGORY_DATA_PROTECTION_AND_PRIVACY_VIOLATIONS""","""v1""",282
"""STATEMENT_CATEGORY_DATA_PROTECTION_AND_PRIVACY_VIOLATIONS""","""v2""",185
"""STATEMENT_CATEGORY_ILLEGAL_OR_HARMFUL_SPEECH""","""v1""",167
"""STATEMENT_CATEGORY_ILLEGAL_OR_HARMFUL_SPEECH""","""v2""",125
"""STATEMENT_CATEGORY_INTELLECTUAL_PROPERTY_INFRINGEMENTS""","""v2""",1018


## 1b. How Many Unique Categories Exist Per Platform?

Before harmonization, both platforms report violations using DSA-defined category labels. The label set changed between v1 (before July 1, 2025) and v2 (from July 1, 2025 onwards). The cell below counts how many distinct category labels each platform actually used in each API version.

In [ ]:
for label, df in [("TikTok", raw_cats), ("X", raw_cats_x)]:
    print(f"{label}:")
    summary = (
        df.group_by("api_version")
        .agg(pl.col("category").n_unique().alias("unique_categories"))
        .sort("api_version")
    )
    for row in summary.iter_rows(named=True):
        print(f"  {row['api_version']}: {row['unique_categories']} unique category labels")
    print()

TikTok:
  v1: 14 unique category labels
  v2: 15 unique category labels

X:
  v1: 16 unique category labels
  v2: 12 unique category labels



## 1c. What Changed Between v1 and v2?

Categories that appear **only in v1** were either removed or renamed in v2. Categories **only in v2** are either new or renamed from v1. Categories **in both** were carried over unchanged. This comparison -- done per platform -- shows exactly why a mapping layer is needed before any cross-period analysis.

In [ ]:
for label, df in [("TikTok", raw_cats), ("X", raw_cats_x)]:
    v1 = set(df.filter(pl.col("api_version") == "v1")["category"].to_list())
    v2 = set(df.filter(pl.col("api_version") == "v2")["category"].to_list())

    only_v1 = sorted(v1 - v2)
    only_v2 = sorted(v2 - v1)
    in_both = sorted(v1 & v2)

    print(f"{'=' * 65}")
    print(f"  {label}")
    print(f"{'=' * 65}")
    print(f"\n  Only in v1 -- removed or renamed in v2 ({len(only_v1)}):")
    for c in only_v1:
        print(f"    - {c}")
    print(f"\n  Only in v2 -- new or renamed from v1 ({len(only_v2)}):")
    for c in only_v2:
        print(f"    - {c}")
    print(f"\n  In both v1 and v2 -- unchanged ({len(in_both)}):")
    for c in in_both:
        print(f"    - {c}")
    print()

  TikTok

  Only in v1 -- removed or renamed in v2 (2):
    - STATEMENT_CATEGORY_NON_CONSENSUAL_BEHAVIOUR
    - STATEMENT_CATEGORY_UNSAFE_AND_ILLEGAL_PRODUCTS

  Only in v2 -- new or renamed from v1 (3):
    - STATEMENT_CATEGORY_CYBER_VIOLENCE
    - STATEMENT_CATEGORY_OTHER_VIOLATION_TC
    - STATEMENT_CATEGORY_UNSAFE_AND_PROHIBITED_PRODUCTS

  In both v1 and v2 -- unchanged (12):
    - STATEMENT_CATEGORY_ANIMAL_WELFARE
    - STATEMENT_CATEGORY_DATA_PROTECTION_AND_PRIVACY_VIOLATIONS
    - STATEMENT_CATEGORY_ILLEGAL_OR_HARMFUL_SPEECH
    - STATEMENT_CATEGORY_INTELLECTUAL_PROPERTY_INFRINGEMENTS
    - STATEMENT_CATEGORY_NEGATIVE_EFFECTS_ON_CIVIC_DISCOURSE_OR_ELECTIONS
    - STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT
    - STATEMENT_CATEGORY_PROTECTION_OF_MINORS
    - STATEMENT_CATEGORY_RISK_FOR_PUBLIC_SECURITY
    - STATEMENT_CATEGORY_SCAMS_AND_FRAUD
    - STATEMENT_CATEGORY_SCOPE_OF_PLATFORM_SERVICE
    - STATEMENT_CATEGORY_SELF_HARM
    - STATEMENT_CATEGORY_VIOLENCE

  X

  On

## 1d. Which Categories Overlap Between TikTok and X?

Across both API versions, TikTok and X draw from the same DSA-defined category vocabulary -- but they do not necessarily use the same labels. The comparison below shows which raw category labels appear in **both** platforms, which are **TikTok-only**, and which are **X-only**. This is a precondition check: categories that neither platform shares cannot be compared cross-platform at the raw label level (only at the harmonized Super-Cluster level).

In [ ]:
tiktok_all = set(raw_cats["category"].to_list())
x_all = set(raw_cats_x["category"].to_list())

in_both = sorted(tiktok_all & x_all)
tiktok_only = sorted(tiktok_all - x_all)
x_only = sorted(x_all - tiktok_all)

print(f"{'=' * 65}")
print(f"  Cross-platform raw category overlap")
print(f"{'=' * 65}")

print(f"\n  In both TikTok and X ({len(in_both)}):")
for c in in_both:
    print(f"    - {c}")

print(f"\n  TikTok only -- not reported by X ({len(tiktok_only)}):")
for c in tiktok_only:
    print(f"    - {c}")

print(f"\n  X only -- not reported by TikTok ({len(x_only)}):")
for c in x_only:
    print(f"    - {c}")

  Cross-platform raw category overlap

  In both TikTok and X (16):
    - STATEMENT_CATEGORY_ANIMAL_WELFARE
    - STATEMENT_CATEGORY_CYBER_VIOLENCE
    - STATEMENT_CATEGORY_DATA_PROTECTION_AND_PRIVACY_VIOLATIONS
    - STATEMENT_CATEGORY_ILLEGAL_OR_HARMFUL_SPEECH
    - STATEMENT_CATEGORY_INTELLECTUAL_PROPERTY_INFRINGEMENTS
    - STATEMENT_CATEGORY_NON_CONSENSUAL_BEHAVIOUR
    - STATEMENT_CATEGORY_OTHER_VIOLATION_TC
    - STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT
    - STATEMENT_CATEGORY_PROTECTION_OF_MINORS
    - STATEMENT_CATEGORY_RISK_FOR_PUBLIC_SECURITY
    - STATEMENT_CATEGORY_SCAMS_AND_FRAUD
    - STATEMENT_CATEGORY_SCOPE_OF_PLATFORM_SERVICE
    - STATEMENT_CATEGORY_SELF_HARM
    - STATEMENT_CATEGORY_UNSAFE_AND_ILLEGAL_PRODUCTS
    - STATEMENT_CATEGORY_UNSAFE_AND_PROHIBITED_PRODUCTS
    - STATEMENT_CATEGORY_VIOLENCE

  TikTok only -- not reported by X (1):
    - STATEMENT_CATEGORY_NEGATIVE_EFFECTS_ON_CIVIC_DISCOURSE_OR_ELECTIONS

  X only -- not reported by TikTok (0):


## 2. Taxonomy Mapping Table

A mapping table (`03_harmonization/taxonomy_mapping_v1_v2.csv`) was created to assign each raw category label to a **Super-Cluster** -- a stable, version-independent label that works across both v1 and v2 records.

The table below shows every original DSA category alongside which platform(s) actually use it, grouped by Super-Cluster. 

- **TikTok_original_category** -- the category label as used in TikTok data (null if TikTok never reported it)
- **X_original_category** -- the category label as used in X data (null if X never reported it)
- **api_version** -- which API versions the category is valid in (`both`, `v1_only`)
- **mapping_type** -- the relationship between the raw label and its Super-Cluster

## 3. Super-Cluster Taxonomy

13 Super-Clusters were defined. Two mapping decisions required explicit justification:

| Decision | Rationale |
|---|---|
| `RISK_FOR_PUBLIC_SECURITY` grouped into **VIOLENCE** | Both involve physical harm. Semantic proximity justifies merging. |
| `CYBER_VIOLENCE` kept as **separate Super-Cluster** | Represents psychological/reputational harm, distinct from physical violence. Also analytically important as a v2-introduced category with a dramatic rise in X data. |
| `NON_CONSENSUAL_BEHAVIOUR` (v1 only) grouped into **SEXUAL_CONTENT** | No v2 equivalent exists. Closest semantic match is SEXUAL_CONTENT. |

All 13 Super-Clusters:

In [ ]:
mapping = pl.read_csv(str(MAPPING_CSV))
super_clusters = mapping.select("super_cluster").unique().sort("super_cluster")
for sc in super_clusters.to_series().to_list():
    print(f"  - {sc}")

  - ANIMAL_WELFARE
  - CIVIC_AND_ELECTIONS
  - CYBER_VIOLENCE
  - HARMFUL_SPEECH
  - INTELLECTUAL_PROPERTY
  - OTHER
  - PRIVACY_AND_DATA
  - PROTECTION_OF_MINORS
  - SCAMS_AND_FRAUD
  - SELF_HARM
  - SEXUAL_CONTENT
  - UNSAFE_PRODUCTS
  - VIOLENCE


In [ ]:
mapping = pl.read_csv(str(MAPPING_CSV))

# Categories actually used by each platform (from raw data computed above)
tiktok_used = set(raw_cats["category"].unique().to_list())
x_used = set(raw_cats_x["category"].unique().to_list())

comparison = (
    mapping
    .with_columns([
        pl.when(pl.col("original_category").is_in(list(tiktok_used)))
          .then(pl.col("original_category"))
          .otherwise(None)
          .alias("tiktok_original_category"),
        pl.when(pl.col("original_category").is_in(list(x_used)))
          .then(pl.col("original_category"))
          .otherwise(None)
          .alias("x_original_category"),
    ])
    .select([
        pl.col("api_version"),
        pl.col("mapping_type"),
        pl.col("tiktok_original_category"),
        pl.col("super_cluster"),
        pl.col("x_original_category"),
        pl.col("mapping_type").alias("mapping_type "),
        pl.col("api_version").alias("api_version "),
    ])
    .sort(["super_cluster", "api_version"])
)

with pl.Config(fmt_str_lengths=80, tbl_rows=50):
    display(comparison)

api_version,mapping_type,tiktok_original_category,super_cluster,x_original_category,mapping_type,api_version
str,str,str,str,str,str,str
"""both""","""direct""","""STATEMENT_CATEGORY_ANIMAL_WELFARE""","""ANIMAL_WELFARE""","""STATEMENT_CATEGORY_ANIMAL_WELFARE""","""direct""","""both"""
"""both""","""direct""","""STATEMENT_CATEGORY_NEGATIVE_EFFECTS_ON_CIVIC_DISCOURSE_OR_ELECTIONS""","""CIVIC_AND_ELECTIONS""",null,"""direct""","""both"""
"""both""","""present_in_both""","""STATEMENT_CATEGORY_CYBER_VIOLENCE""","""CYBER_VIOLENCE""","""STATEMENT_CATEGORY_CYBER_VIOLENCE""","""present_in_both""","""both"""
"""both""","""direct""","""STATEMENT_CATEGORY_ILLEGAL_OR_HARMFUL_SPEECH""","""HARMFUL_SPEECH""","""STATEMENT_CATEGORY_ILLEGAL_OR_HARMFUL_SPEECH""","""direct""","""both"""
"""both""","""direct""","""STATEMENT_CATEGORY_INTELLECTUAL_PROPERTY_INFRINGEMENTS""","""INTELLECTUAL_PROPERTY""","""STATEMENT_CATEGORY_INTELLECTUAL_PROPERTY_INFRINGEMENTS""","""direct""","""both"""
"""both""","""renamed_from_v1""","""STATEMENT_CATEGORY_OTHER_VIOLATION_TC""","""OTHER""","""STATEMENT_CATEGORY_OTHER_VIOLATION_TC""","""renamed_from_v1""","""both"""
"""both""","""renamed_to_v2""","""STATEMENT_CATEGORY_SCOPE_OF_PLATFORM_SERVICE""","""OTHER""","""STATEMENT_CATEGORY_SCOPE_OF_PLATFORM_SERVICE""","""renamed_to_v2""","""both"""
"""both""","""direct""","""STATEMENT_CATEGORY_DATA_PROTECTION_AND_PRIVACY_VIOLATIONS""","""PRIVACY_AND_DATA""","""STATEMENT_CATEGORY_DATA_PROTECTION_AND_PRIVACY_VIOLATIONS""","""direct""","""both"""
"""both""","""direct""","""STATEMENT_CATEGORY_PROTECTION_OF_MINORS""","""PROTECTION_OF_MINORS""","""STATEMENT_CATEGORY_PROTECTION_OF_MINORS""","""direct""","""both"""


## 3c. Which Original Categories Map to Each Super-Cluster -- Per Platform?

Not all DSA categories appear in both platforms. The table below joins each platform's actual raw category records with the mapping table to show how many records fall under each Super-Cluster per API version. This reveals which Super-Clusters are analytically meaningful for each platform.

In [ ]:
for label, df in [("TikTok", raw_cats), ("X", raw_cats_x)]:
    breakdown = (
        df
        .join(
            mapping.select(["original_category", "super_cluster"]),
            left_on="category",
            right_on="original_category",
            how="left"
        )
        .group_by(["super_cluster", "api_version"])
        .agg(pl.col("count").sum().alias("total_records"))
        .sort(["super_cluster", "api_version"])
    )
    print(f"\n{label} -- records per Super-Cluster per api_version:")
    with pl.Config(fmt_str_lengths=60, tbl_rows=50):
        display(breakdown)


TikTok -- records per Super-Cluster per api_version:


super_cluster,api_version,total_records
str,str,u32
"""ANIMAL_WELFARE""","""v1""",948719
"""ANIMAL_WELFARE""","""v2""",198251
"""CIVIC_AND_ELECTIONS""","""v1""",37292141
"""CIVIC_AND_ELECTIONS""","""v2""",8247264
"""CYBER_VIOLENCE""","""v2""",1389
"""HARMFUL_SPEECH""","""v1""",399152169
"""HARMFUL_SPEECH""","""v2""",35245452
"""INTELLECTUAL_PROPERTY""","""v1""",45491
"""INTELLECTUAL_PROPERTY""","""v2""",94441



X -- records per Super-Cluster per api_version:


super_cluster,api_version,total_records
str,str,u32
"""ANIMAL_WELFARE""","""v1""",10
"""ANIMAL_WELFARE""","""v2""",3
"""CYBER_VIOLENCE""","""v1""",34
"""CYBER_VIOLENCE""","""v2""",19143
"""HARMFUL_SPEECH""","""v1""",167
"""HARMFUL_SPEECH""","""v2""",125
"""INTELLECTUAL_PROPERTY""","""v1""",906
"""INTELLECTUAL_PROPERTY""","""v2""",1018
"""OTHER""","""v1""",60975


## 3d. Schema Anomalies

Two categories appeared outside their officially defined API version scope. These are real platform submissions using an unexpected label -- not data errors. Both are documented in the cleaning decisions log (`CD-20260429-02`) and preserved in `category_raw`. The `category_harmonized` column handles them correctly via the mapping table.

| Category | Expected version | Found in | Interpretation |
|---|---|---|---|
| `STATEMENT_CATEGORY_CYBER_VIOLENCE` | v2 only | v1 records | Platforms submitted this label before it was officially introduced |
| `STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT` | v1 only | v2 records | Platforms continued using a removed label after the v2 transition |

In [ ]:
anomalies = [
    ("STATEMENT_CATEGORY_CYBER_VIOLENCE", "v1", "officially a v2-only category"),
    ("STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT", "v2", "officially removed in v2"),
]

for label, raw_df in [("TikTok", raw_cats), ("X", raw_cats_x)]:
    print(f"{label}:")
    for category, version, note in anomalies:
        match = raw_df.filter(
            (pl.col("api_version") == version) & (pl.col("category") == category)
        )
        count = match["count"].sum() if len(match) > 0 else 0
        print(f"  {category}")
        print(f"    found in {version} ({note}): {count:,} records")
    print()

TikTok:
  STATEMENT_CATEGORY_CYBER_VIOLENCE
    found in v1 (officially a v2-only category): 0 records
  STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT
    found in v2 (officially removed in v2): 7 records

X:
  STATEMENT_CATEGORY_CYBER_VIOLENCE
    found in v1 (officially a v2-only category): 34 records
  STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT
    found in v2 (officially removed in v2): 0 records



## 4. Category Distribution After Harmonization

With categories harmonized, distribution can be compared across v1 and v2 and across platforms.

In [ ]:
# TikTok: category distribution per api_version
tiktok_dist = (
    pl.scan_parquet(str(TIKTOK_HARMONIZED))
    .group_by([ "category_harmonized"])
    .agg(pl.len().alias("count"))
    .sort(["count"], descending=[True])
    .collect(engine="streaming")
)
print("TikTok -- harmonized category distribution:")

with pl.Config(fmt_str_lengths=100, tbl_rows=100):
    display(tiktok_dist)

TikTok -- harmonized category distribution:


category_harmonized,count
str,u32
"""OTHER""",441285426
"""HARMFUL_SPEECH""",434397621
"""CIVIC_AND_ELECTIONS""",45539405
"""VIOLENCE""",33986154
"""PROTECTION_OF_MINORS""",16073771
"""SCAMS_AND_FRAUD""",13559912
"""SEXUAL_CONTENT""",5716920
"""PRIVACY_AND_DATA""",4599648
"""SELF_HARM""",1240313


In [ ]:
# X: category distribution per api_version
x_dist = (
    pl.scan_parquet(str(X_HARMONIZED))
    .group_by([ "category_harmonized"])
    .agg(pl.len().alias("count"))
    .sort(["count"], descending=[True])
    .collect()
)
print("X -- harmonized category distribution:")
with pl.Config(fmt_str_lengths=100, tbl_rows=100):
    display(x_dist)

X -- harmonized category distribution:


category_harmonized,count
str,u32
"""OTHER""",68240
"""SEXUAL_CONTENT""",34292
"""PROTECTION_OF_MINORS""",20753
"""CYBER_VIOLENCE""",19177
"""SCAMS_AND_FRAUD""",13226
"""VIOLENCE""",10954
"""UNSAFE_PRODUCTS""",10881
"""SELF_HARM""",3102
"""INTELLECTUAL_PROPERTY""",1924


## 5. Sample of Final Harmonized Data

The harmonization script added two new columns to each platform's working file:

- `category_harmonized` -- the Super-Cluster label


 

The cells below show the first 8 rows of the final harmonized parquet for each platform. All analysis-ready columns are visible: the original raw values are preserved alongside the harmonized counterparts.

In [ ]:
print("TikTok -- harmonized data sample:")
(
    pl.scan_parquet(str(TIKTOK_HARMONIZED))
    .head(8)
    .collect()
)

TikTok -- harmonized data sample:


uuid,content_type,automated_detection,automated_decision,territorial_scope,platform_name,decision_visibility,api_version,category_raw,category_harmonized,application_date
str,str,str,str,str,str,str,str,str,str,date
"""0a5b1861-9688-4573-a189-95bc32…","""[""CONTENT_TYPE_TEXT""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…","""TikTok""","""[""DECISION_VISIBILITY_CONTENT_…","""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01
"""be715329-98ea-470b-b60a-8ca47e…","""[""CONTENT_TYPE_TEXT""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…","""TikTok""","""[""DECISION_VISIBILITY_CONTENT_…","""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01
"""f3e04b02-5c6c-4686-a040-3c707f…","""[""CONTENT_TYPE_TEXT""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…","""TikTok""","""[""DECISION_VISIBILITY_CONTENT_…","""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01
"""525c072f-1890-44d0-8cf2-849b2f…","""[""CONTENT_TYPE_TEXT""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…","""TikTok""","""[""DECISION_VISIBILITY_CONTENT_…","""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01
"""78ad61f3-b271-484c-ba05-ad6198…","""[""CONTENT_TYPE_VIDEO""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…","""TikTok""","""[""DECISION_VISIBILITY_CONTENT_…","""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01
"""e6b11bc8-aacb-47b7-80c9-a5275b…","""[""CONTENT_TYPE_TEXT""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…","""TikTok""","""[""DECISION_VISIBILITY_CONTENT_…","""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01
"""e1fc71df-1c91-4bad-977e-98619c…","""[""CONTENT_TYPE_TEXT""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…","""TikTok""","""[""DECISION_VISIBILITY_CONTENT_…","""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01
"""b8f9f6f2-4620-4481-87b1-cc3b23…","""[""CONTENT_TYPE_TEXT""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…","""TikTok""","""[""DECISION_VISIBILITY_CONTENT_…","""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01


In [ ]:
print("X -- harmonized data sample:")
(
    pl.scan_parquet(str(X_HARMONIZED))
    .head(8)
    .collect()
)

X -- harmonized data sample:


uuid,content_type,automated_detection,automated_decision,territorial_scope,platform_name,decision_visibility,api_version,category_raw,category_harmonized,application_date
str,str,str,str,str,str,str,str,str,str,date
"""87fb4591-c1e7-4ed7-b32d-a24f3d…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""","""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v2""","""STATEMENT_CATEGORY_UNSAFE_AND_…","""UNSAFE_PRODUCTS""",2025-07-06
"""1689be96-2486-4d59-99a8-7d211c…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""","""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v2""","""STATEMENT_CATEGORY_UNSAFE_AND_…","""UNSAFE_PRODUCTS""",2025-07-06
"""eab8c83b-02ca-473d-9e29-649bd6…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""","""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v2""","""STATEMENT_CATEGORY_UNSAFE_AND_…","""UNSAFE_PRODUCTS""",2025-07-06
"""fffa15e1-257d-4f18-a0a6-05d51c…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""","""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v2""","""STATEMENT_CATEGORY_UNSAFE_AND_…","""UNSAFE_PRODUCTS""",2025-07-06
"""2f3895a9-93b7-40bd-85a2-e2fdd8…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""","""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v2""","""STATEMENT_CATEGORY_UNSAFE_AND_…","""UNSAFE_PRODUCTS""",2025-07-06
"""3b443fc7-a661-44e0-a859-a2dc4e…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""","""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v2""","""STATEMENT_CATEGORY_SCAMS_AND_F…","""SCAMS_AND_FRAUD""",2025-07-06
"""3bd0d063-8f4c-4850-8619-b2e8ea…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""","""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v2""","""STATEMENT_CATEGORY_UNSAFE_AND_…","""UNSAFE_PRODUCTS""",2025-07-06
"""05c102c6-455d-46c5-82f5-db7763…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""","""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v2""","""STATEMENT_CATEGORY_UNSAFE_AND_…","""UNSAFE_PRODUCTS""",2025-07-06


## 6. Key Findings from Harmonization

- **TikTok dominant category shifted dramatically:** `HARMFUL_SPEECH` dominated v1 (48.05%), but `OTHER` (mapping from `OTHER_VIOLATION_TC`) dominated v2 (67.27%). This reflects a genuine change in TikTok's reporting behavior after the schema transition, not a data artifact.
- **X CYBER_VIOLENCE exploded in v2:** 34 records in v1 vs 19,143 records in v2 (37.44% of all X v2 records). The clearest schema transition effect in the dataset.
- **Zero unmapped categories:** Validation confirmed every raw category label in both platforms was covered by the mapping table.